In [ ]:
import json
import os
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    TrainingArguments, 
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training, PeftModel
import numpy as np
from sklearn.model_selection import train_test_split
import gc

def load_direct_dataset_from_csv(csv_file_path):
    """Load the simplified dataset from CSV file"""
    try:
        df = pd.read_csv(csv_file_path)
        
        # Assuming columns are: disease_name, clinical_note, reasoning_chain
        # Check if we have the expected columns
        if len(df.columns) < 3:
            print(f"Warning: CSV file has only {len(df.columns)} columns, expected at least 3")
            
        # Map columns based on expected structure
        # Column 0: disease name, Column 1: clinical note, Column 2: reasoning chain
        if len(df.columns) >= 3:
            df = df.iloc[:, :3]  # Take only first 3 columns
            df.columns = ['disease', 'clinical_note', 'reasoning_chain']
        else:
            # Try to handle with fewer columns
            print("Adjusting column names based on available data...")
            if len(df.columns) == 1:
                df['clinical_note'] = ''
                df['reasoning_chain'] = ''
            elif len(df.columns) == 2:
                df.columns = ['disease', 'clinical_note']
                df['reasoning_chain'] = ''
            else:
                df.columns = ['disease', 'clinical_note', 'reasoning_chain'][:len(df.columns)]
        
        # Clean the data
        df = df.dropna(subset=['disease', 'reasoning_chain'])
        df = df[df['reasoning_chain'].str.strip() != '']
        df = df[df['disease'].str.strip() != '']
        
        # Fill empty clinical notes if needed
        df['clinical_note'] = df['clinical_note'].fillna('').astype(str)
        
        print(f"Loaded {len(df)} examples from CSV")
        return df
        
    except Exception as e:
        print(f"Error loading CSV file: {e}")
        return pd.DataFrame()

def load_model_with_frozen_adapters():
    """Load model with frozen phase 1 adapters using the SAME tokenizer from phase 1"""
    
    # Load the tokenizer from phase 1 to ensure consistency
    print("Loading tokenizer from phase 1...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(
            "./qwen1.5b-symptoms-precautions",
            trust_remote_code=True,
            use_fast=True
        )
        print("Successfully loaded tokenizer from phase 1")
    except:
        print("Failed to load tokenizer from phase 1, using base model tokenizer")
        tokenizer = AutoTokenizer.from_pretrained(
            "Qwen/Qwen2.5-1.5B-Instruct",
            trust_remote_code=True,
            use_fast=True
        )
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    
    # Load base model with quantization - use the same config as phase 1
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    
    print("Loading base model...")
    base_model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen2.5-1.5B-Instruct",
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    
    # First, resize the model to match the tokenizer
    base_model.resize_token_embeddings(len(tokenizer))
    
    # Now load phase 1 adapters and freeze them
    print("Loading and freezing phase 1 adapters...")
    try:
        model = PeftModel.from_pretrained(
            base_model,
            "./qwen1.5b-symptoms-precautions/phase1_adapters",
            is_trainable=False  # Freeze phase 1 adapters
        )
        print("Successfully loaded phase 1 adapters")
    except Exception as e:
        print(f"Error loading phase 1 adapters: {e}")
        print("Continuing without phase 1 adapters...")
        model = base_model
    
    # Prepare model for k-bit training
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    
    # Add new LoRA adapters for phase 2 with different target modules
    lora_config_phase2 = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=4,
        lora_alpha=8,
        lora_dropout=0.05,
        target_modules=["gate_proj", "up_proj"],
        bias="none",
    )
    
    model = get_peft_model(model, lora_config_phase2)
    
    print("\nTrainable parameters (Phase 2 only):")
    model.print_trainable_parameters()
    
    return model, tokenizer

class CustomDataCollator:
    """Custom data collator that handles padding properly for variable length sequences"""
    def __init__(self, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __call__(self, features):
        # Separate input_ids, attention_mask, and labels
        input_ids = [feature['input_ids'] for feature in features]
        labels = [feature['labels'] for feature in features]
        
        # Find max length in this batch
        batch_max_length = max(len(seq) for seq in input_ids)
        batch_max_length = min(batch_max_length, self.max_length)
        
        # Pad sequences
        padded_input_ids = []
        padded_attention_mask = []
        padded_labels = []
        
        for i in range(len(input_ids)):
            input_seq = input_ids[i]
            label_seq = labels[i]
            
            # Truncate if too long
            if len(input_seq) > batch_max_length:
                input_seq = input_seq[:batch_max_length]
                label_seq = label_seq[:batch_max_length]
            
            # Pad sequences
            pad_length = batch_max_length - len(input_seq)
            
            padded_input = input_seq + [self.tokenizer.pad_token_id] * pad_length
            padded_attention = [1] * len(input_seq) + [0] * pad_length
            padded_label = label_seq + [-100] * pad_length  # Use -100 for padding in labels
            
            padded_input_ids.append(padded_input)
            padded_attention_mask.append(padded_attention)
            padded_labels.append(padded_label)
        
        # Convert to tensors
        batch = {
            'input_ids': torch.tensor(padded_input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(padded_attention_mask, dtype=torch.long),
            'labels': torch.tensor(padded_labels, dtype=torch.long),
        }
        
        return batch

def compute_metrics_for_evaluation(pred):
    """Custom metrics computation focusing only on disease and reasoning chain"""
    # This is a placeholder - actual implementation would need more complex evaluation
    # For now, we'll just return the loss as computed by the model
    
    # The Trainer already computes loss during evaluation
    # We can extract it from the predictions if needed
    # Here we're just returning an empty dict to satisfy the trainer
    return {}

def preprocess_reasoning_function(examples, tokenizer, is_training=True):
    """Preprocess reasoning data focusing only on disease and reasoning chain"""
    texts = []
    
    for i in range(len(examples['clinical_note'])):
        clinical_note = examples['clinical_note'][i]
        reasoning_chain = examples['reasoning_chain'][i]
        disease = examples['disease'][i]
        
        # Truncate clinical note to save memory
        clinical_note = clinical_note[:1500]
        
        # Simplified prompt format focused only on what we're evaluating
        if is_training:
            # For training: provide full context and expected output
            instruction = """Analyze this clinical note and provide:
1. The most suspected disease
2. The diagnostic reasoning behind this conclusion

Please structure your response as follows:
**Most Suspected Disease:** [disease name]
**Diagnostic Reasoning:** [your reasoning chain]"""
            
            target_response = f"""**Most Suspected Disease:** {disease}
**Diagnostic Reasoning:** {reasoning_chain}"""
        else:
            # For evaluation: simpler instruction
            instruction = """Based on the clinical note, identify the most suspected disease and explain your reasoning."""
            
            # During evaluation, we don't provide the target - model should generate it
            target_response = ""
        
        full_text = f"<|im_start|>user\n{instruction}\n\nClinical Note:\n{clinical_note}<|im_end|>\n<|im_start|>assistant\n{target_response}<|im_end|>"
        texts.append(full_text)
    
    # Tokenize WITH truncation to prevent OOM
    tokenized = tokenizer(
        texts,
        truncation=True,
        padding=False,
        max_length=512,
        return_tensors=None,
        add_special_tokens=True,
    )
    
    # Create labels - for causal LM, labels are the same as input_ids
    result = {
        'input_ids': tokenized['input_ids'],
        'labels': tokenized['input_ids'].copy()
    }
    
    return result

class CustomTrainer(Trainer):
    """Custom trainer to focus evaluation on disease and reasoning chain"""
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """
        Override compute_loss to ensure we're only evaluating the relevant parts.
        The model's internal loss computation already handles this via the labels tensor.
        """
        # Call parent's compute_loss - this already uses the labels we provided
        return super().compute_loss(model, inputs, return_outputs, num_items_in_batch)

def print_gpu_memory():
    """Print current GPU memory usage"""
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
            print(f"  Allocated: {torch.cuda.memory_allocated(i) / 1024**3:.2f} GB")
            print(f"  Reserved: {torch.cuda.memory_reserved(i) / 1024**3:.2f} GB")

def main():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
    
    print("="*80)
    print("PHASE 2: CLINICAL REASONING TRAINING (FOCUSED ON DISEASE & REASONING)")
    print("="*80)
    
    # Check if phase 1 adapters exist
    if not os.path.exists("./qwen1.5b-symptoms-precautions/phase1_adapters"):
        print("WARNING: Phase 1 adapters not found!")
        print("Please run phase 1 training first.")
        print("Continuing without phase 1 adapters...")
    
    # Load reasoning dataset from CSV
    print("\nLoading reasoning dataset from CSV...")
    csv_file_path = "direct_dataset_simplified.csv"
    df = load_direct_dataset_from_csv(csv_file_path)
    
    if len(df) == 0:
        print(f"No reasoning data found in {csv_file_path}!")
        print("Please check the CSV file exists and has the correct format.")
        print("Expected format: disease_name, clinical_note, reasoning_chain")
        return
    
    print(f"Reasoning examples loaded: {len(df)}")
    print(f"Unique diseases: {df['disease'].nunique()}")
    
    # Show sample of the data structure
    print(f"\nSample data structure:")
    for i in range(min(2, len(df))):
        print(f"\n--- Sample {i+1} ---")
        print(f"Disease: {df.iloc[i]['disease']}")
        print(f"Clinical Note (first 200 chars): {df.iloc[i]['clinical_note'][:200]}...")
        print(f"Reasoning Chain: {df.iloc[i]['reasoning_chain']}")
    
    # Split data
    train_df, eval_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['disease'])
    
    print(f"\nTraining set: {len(train_df)} examples")
    print(f"Validation set: {len(eval_df)} examples")
    
    # Create datasets
    train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
    eval_dataset = Dataset.from_pandas(eval_df.reset_index(drop=True))
    
    dataset_dict = DatasetDict({
        "train": train_dataset,
        "validation": eval_dataset
    })
    
    # Load model with frozen phase 1 adapters
    print("\nLoading model with frozen phase 1 adapters...")
    model, tokenizer = load_model_with_frozen_adapters()
    
    # Preprocess data - separate processing for train and eval
    def preprocess_train_batch(examples):
        return preprocess_reasoning_function(examples, tokenizer, is_training=True)
    
    def preprocess_eval_batch(examples):
        return preprocess_reasoning_function(examples, tokenizer, is_training=False)
    
    print("\nTokenizing datasets (with truncation)...")
    tokenized_train = dataset_dict["train"].map(
        preprocess_train_batch,
        batched=True,
        remove_columns=dataset_dict["train"].column_names,
        desc="Tokenizing training data",
    )
    
    tokenized_val = dataset_dict["validation"].map(
        preprocess_eval_batch,
        batched=True,
        remove_columns=dataset_dict["validation"].column_names,
        desc="Tokenizing validation data",
    )
    
    tokenized_datasets = DatasetDict({
        "train": tokenized_train,
        "validation": tokenized_val
    })
    
    print(f"Tokenized training examples: {len(tokenized_datasets['train'])}")
    print(f"Tokenized validation examples: {len(tokenized_datasets['validation'])}")
    
    # Check sequence lengths
    train_lengths = [len(x['input_ids']) for x in tokenized_datasets['train']]
    val_lengths = [len(x['input_ids']) for x in tokenized_datasets['validation']]
    print(f"\nSequence length stats (after truncation):")
    print(f"Train - Min: {min(train_lengths)}, Max: {max(train_lengths)}, Mean: {np.mean(train_lengths):.1f}")
    print(f"Val   - Min: {min(val_lengths)}, Max: {max(val_lengths)}, Mean: {np.mean(val_lengths):.1f}")
    
    # Use custom data collator
    data_collator = CustomDataCollator(tokenizer, max_length=512)
    
    # Training arguments for phase 2 - focused evaluation
    training_args = TrainingArguments(
        output_dir="./qwen1.5b-clinical-reasoning-focused",
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=1e-5,
        num_train_epochs=3,
        logging_dir="./logs",
        logging_steps=5,
        eval_steps=50,
        save_steps=100,
        eval_strategy="steps",
        save_strategy="steps",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        warmup_steps=20,
        fp16=True,
        dataloader_pin_memory=False,
        save_total_limit=1,
        remove_unused_columns=False,
        report_to="none",
        optim="paged_adamw_8bit",
        max_grad_norm=0.3,
        dataloader_drop_last=True,
        gradient_checkpointing=True,
        eval_accumulation_steps=2,
        # Add evaluation settings
        evaluation_strategy="steps",
        prediction_loss_only=True,  # Only compute loss during evaluation
    )
    
    # Create custom trainer
    trainer = CustomTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        data_collator=data_collator,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics_for_evaluation,
    )
    
    # Train phase 2
    print("\n" + "="*80)
    print("STARTING PHASE 2 TRAINING (FOCUSED ON DISEASE & REASONING)")
    print("="*80)
    print(f"Focus: Disease identification and reasoning chain only")
    print(f"Data source: {csv_file_path}")
    print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
    
    try:
        train_result = trainer.train()
        
        # Save final model
        print("\nSaving final model...")
        trainer.save_model()
        tokenizer.save_pretrained("./qwen1.5b-clinical-reasoning-focused")
        
        # Save both adapters
        model.save_pretrained("./qwen1.5b-clinical-reasoning-focused/final_adapters")
        
        # Print training metrics
        print(f"\nTraining metrics:")
        for key, value in train_result.metrics.items():
            print(f"  {key}: {value:.4f}")
        
        print(f"\n{'='*80}")
        print("TRAINING COMPLETED SUCCESSFULLY!")
        print(f"{'='*80}")
        print("Final model saved in: ./qwen1.5b-clinical-reasoning-focused")
        print("Final adapters saved in: ./qwen1.5b-clinical-reasoning-focused/final_adapters")
        print("\nNote: Evaluation loss focuses only on disease and reasoning chain prediction.")
        
    except Exception as e:
        print(f"Training failed with error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()